### Работа с вебхуками и polling

Современные веб-приложения и сервисы используют два основных подхода для получения данных от клиента или внешних систем:
1. **Polling** (опрашивание) — периодическое отправление запросов для проверки наличия новых данных.
2. **Webhook** (вебхук) — механизм, при котором сервер уведомляет клиента о событиях в реальном времени.

Эти подходы имеют ключевое значение для разработки ботов, API-интеграций и серверных приложений. Для их эффективного применения нужно понимать, как работает инфраструктура серверов (WSGI, веб-серверы, и их взаимодействие).



### Polling: Опрашивание сервера

**Polling** — это процесс, при котором клиент регулярно отправляет запросы на сервер, чтобы проверить наличие новых данных.

#### Как работает polling?

1. Клиент отправляет запрос на сервер (например, каждую секунду).
2. Сервер обрабатывает запрос и возвращает данные, если они есть.
3. Если данных нет, сервер возвращает пустой ответ.
4. Клиент повторяет процесс через заданный интервал времени.

#### Пример polling в Telegram-ботах

В Telegram polling реализован через метод **getUpdates**, который возвращает новые сообщения от пользователей.

```plaintext
Клиент → Telegram API: "Есть ли новые сообщения?"
Telegram API → Клиент: "Вот новые сообщения."
```

На уровне кода с использованием библиотеки **Aiogram** это выглядит так:

```python
from aiogram import Bot, Dispatcher
from aiogram.utils import executor

TOKEN = "ваш_токен"
bot = Bot(token=TOKEN)
dp = Dispatcher(bot)

# Стартуем бота с polling
if __name__ == "__main__":
    executor.start_polling(dp, skip_updates=True)
```

#### Преимущества polling

1. Простота реализации.
2. Не требуется сложной инфраструктуры (достаточно клиента, который будет отправлять запросы).

#### Недостатки polling

1. **Высокая нагрузка на сервер**:
   - Постоянные запросы увеличивают потребление ресурсов, особенно если данные обновляются редко.
2. **Задержки**:
   - Интервал между запросами может приводить к задержке получения данных.


### ngrok

**ngrok** — это инструмент, который позволяет создавать защищённые туннели от локального сервера (запущенного на вашем компьютере) к публичному интернету. Это полезно, когда вы хотите протестировать или продемонстрировать локальные веб-приложения, не настраивая сложную инфраструктуру.

https://www.ripe.net/manage-ips-and-asns/db/ipv4-blocks-for-private-internets/ Информация по приватным диапазонам сетевых адресов

https://www.comptia.org/content/guides/what-is-network-address-translation#:~:text=NAT%20stands%20for%20network%20address,as%20do%20most%20home%20routers. - Что такое NAT
#### **Как работает ngrok?**


1. **Установка туннеля**:
   - ngrok запускает туннель на вашем локальном сервере (например, `localhost:3000`) и создаёт публичный URL (например, `https://abc123.ngrok.io`).
   
2. **Перенаправление запросов**:
   - Все запросы, приходящие на публичный URL, перенаправляются на ваш локальный сервер.

3. **Обеспечение HTTPS**:
   - ngrok автоматически предоставляет защищённое HTTPS-соединение для вашего локального сервера, что особенно важно для интеграций с сервисами, требующими HTTPS (например, Telegram webhook).

#### **Почему нужен ngrok?**

1. **Для локальной разработки**:
   - Вы можете разрабатывать приложения на локальном сервере и тестировать их, не размещая на хостинге.
   
2. **Для интеграций с внешними сервисами**:
   - Некоторые сервисы, такие как Telegram, Stripe или GitHub Webhooks, требуют публичного HTTPS-адреса для обратных вызовов. Ngrok делает это возможным.

3. **Для демонстрации работы**:
   - Вы можете быстро показать работу приложения коллегам или клиентам, не публикуя его в интернете.

#### **Пример использования ngrok**

Предположим, у вас есть локальный сервер, работающий на `localhost:3000`. Вы хотите сделать его доступным через интернет.

1. Запустите сервер:
   ```bash
   python your_server.py
   ```

2. Включите ngrok:
   ```bash
   ngrok http 3000
   ```

3. Вы увидите публичный URL, например:
   ```
   https://abc123.ngrok.io
   ```

4. Теперь ваш локальный сервер доступен через этот URL. Например:
   - `http://abc123.ngrok.io`
   - `https://abc123.ngrok.io`

### **Применение в Telegram webhook**

Telegram требует HTTPS для установки webhook. С помощью ngrok это можно сделать следующим образом:

1. Запустите свой Telegram-бот с поддержкой webhook на `localhost:3000`.

2. Включите ngrok:
   ```bash
   ngrok http 3000
   ```

3. Скопируйте полученный HTTPS-URL (например, `https://abc123.ngrok.io`).

4. Установите webhook для вашего бота:
   ```bash
   curl -F "url=https://abc123.ngrok.io/webhook" https://api.telegram.org/bot<ваш_токен>/setWebhook
   ```

5. Теперь Telegram будет отправлять запросы на ваш локальный сервер через публичный URL ngrok.

### Webhook: Реактивная обработка событий

**Webhook** — это обратный вызов, при котором сервер автоматически отправляет данные клиенту при наступлении события.

#### Как работает webhook?

1. Клиент регистрирует URL для получения данных.
2. При наступлении события сервер отправляет HTTP-запрос на этот URL.
3. Клиент обрабатывает запрос и отвечает серверу.

#### Пример webhook в Telegram

В Telegram webhook используется для передачи новых сообщений ботам. Схема взаимодействия:

```plaintext
Telegram API → Сервер: "Вот новое сообщение."
Сервер → Telegram API: "Сообщение обработано."
```

#### Преимущества webhook

1. **Мгновенные уведомления**:
   - Данные отправляются сразу после события.
2. **Экономия ресурсов**:
   - Нет необходимости в постоянных запросах.

#### Недостатки webhook

1. **Сложность настройки**:
   - Необходимо настраивать сервер и обеспечивать его доступность.


In [ ]:
import asyncio
from aiogram import Bot, Dispatcher, Router
from aiogram.filters import Command
from aiogram.types import Message
from aiogram.webhook.aiohttp_server import SimpleRequestHandler
from aiohttp import web

# Конфигурация
BOT_TOKEN = "......"
WEBHOOK_HOST = "https://.....ngrok-free.app"  # Адрес от ngrok
WEBHOOK_PATH = "/webhook"
WEBHOOK_URL = f"{WEBHOOK_HOST}{WEBHOOK_PATH}"
WEBAPP_HOST = "0.0.0.0"
WEBAPP_PORT = 3000

# Инициализация бота и диспетчера
bot = Bot(token=BOT_TOKEN)
dp = Dispatcher()

router = Router()  #  Создаём Router

@router.message(Command("start"))  #  Привязываем обработчик к Router
async def start_command(message: Message):
    await message.answer("Привет! Этот бот использует webhook.")

async def handle_root(request):
    return web.Response(text="Сервер работает!")

# Настройка webhook на старте
async def on_startup():
    await bot.set_webhook(WEBHOOK_URL)
    print(f"Webhook установлен: {WEBHOOK_URL}")

# Удаление webhook при остановке
async def on_shutdown():
    await bot.delete_webhook()
    print("Webhook удалён")
    await bot.session.close()

# Запуск сервера с Aiohttp
async def main():
    app = web.Application()

    # Регистрируем router в Dispatcher
    dp.include_router(router)

    # Регистрация хэндлера для обработки запросов от Telegram
    SimpleRequestHandler(dispatcher=dp, bot=bot).register(app, path=WEBHOOK_PATH)

    # Подключаем хендлеры на старте и при завершении работы
    app.on_startup.append(lambda _: asyncio.create_task(on_startup()))
    app.on_shutdown.append(lambda _: asyncio.create_task(on_shutdown()))

    app.router.add_get("/", handle_root)

    # Запускаем Aiohttp сервер
    runner = web.AppRunner(app)
    await runner.setup()
    site = web.TCPSite(runner, host=WEBAPP_HOST, port=WEBAPP_PORT)
    print(f"Сервер запущен на {WEBAPP_HOST}:{WEBAPP_PORT}")
    await site.start()

    try:
        await asyncio.Event().wait()  # Ожидаем завершения работы сервера
    except asyncio.CancelledError:
        pass
    finally:
        await runner.cleanup()  # Корректно очищаем ресурсы

if __name__ == "__main__":
    asyncio.run(main())


Telegram принимает вебхуки только через HTTPS для защиты передаваемых данных. Для локального обучения можно использовать **ngrok**, который преобразует ваш локальный сервер в публичный HTTPS-адрес.

#### 1. Установите ngrok

- Если его нет, скачайте его с [официального сайта](https://ngrok.com/).

#### 2. Настройка ngrok

1. Запустите локальный сервер с ботом (например, Python-скрипт выше).
2. Включите ngrok:
   ```bash
   ngrok http 3000
   ```
3. Скопируйте публичный HTTPS-адрес, который покажет ngrok, и вставьте его в `WEBHOOK_HOST` (например, `https://abc123.ngrok.io`).

#### 3. Как проверить работу?

1. После запуска бота отправим команду `/start` в Telegram. Мы должны получить ответ: "Привет! Этот бот использует webhook для обработки сообщений". В терминале с ngrok будут видны запросы от Telegram на ваш сервер. Если что-то идёт не так
2. Пробуем `curl http://localhost:3000/webhook` (должна быть ошибка из-за недопустимого метода)
3. Теперь `curl -X POST http://localhost:3000/webhook -H "Content-Type: application/json" -d '{}'` (должно быть ок)
4. Проверим, зарегистрирован ли webhook `curl https://api.telegram.org/bot<ВАШ_ТОКЕН>/getWebhookInfo` (должно быть инфо)
5. Перерегистрируем webhook, если совсем не работает `curl -F "url=https://your-ngrok-url.ngrok-free.app/webhook" https://api.telegram.org/bot<ВАШ_ТОКЕН>/setWebhook` (должно быть Webhook is already set)



#### Что означают параметры и хэндлеры:
   - `WEBHOOK_HOST`: Доменное имя или IP-адрес, доступный через интернет (например, https://example.com).
   - `WEBHOOK_PATH`: Путь для Telegram, куда отправлять запросы (например, `/webhook`).
   - `WEBHOOK_URL`: Полный адрес для webhook (например, https://example.com/webhook).
   - `WEBAPP_HOST`: Локальный хост для запуска приложения (например, `0.0.0.0`).
   - `WEBAPP_PORT`: Порт, на котором сервер будет слушать запросы (например, `3000`).

1. **`handle_root` регистрируется через `app.router.add_get("/", handle_root)`**, потому что это обработчик обычных HTTP-запросов. Он используется в Aiohttp и отвечает на запросы, которые приходят на сервер, например, когда мы проверяем доступность сервера через `curl http://localhost:3000`. Этот обработчик не имеет отношения к Telegram и нужен только для работы веб-сервера.

2. **`start_command` регистрируется через `@dp.message(Command("start"))`**, потому что он предназначен для обработки сообщений в Telegram. Aiogram работает с сообщениями Telegram, которые приходят через webhook, и передаёт их в зарегистрированные обработчики. Когда пользователь отправляет команду `/start`, Telegram передаёт это сообщение через webhook в бота, и Aiogram вызывает `start_command`, если оно было зарегистрировано.


#### Разбор `main()`

Функция `main()` выполняет настройку и запуск веб-сервера на базе `aiohttp`. Этот сервер принимает запросы от Telegram API и передаёт их в `aiogram` для обработки.

#### Шаги в `main()`
```python
app = web.Application()
```
Создаётся веб-приложение Aiohttp. Это объект, который управляет маршрутизацией (какие запросы обрабатывать) и настройками сервера.

```python
dp.include_router(router)
```
Регистрируется `router` в `Dispatcher`.  
В `aiogram 3.x` обработчики команд (`/start`, `/help` и т. д.) теперь добавляются через `Router`.  
Этот шаг связывает их с `Dispatcher`, чтобы Telegram API мог их использовать.

```python
SimpleRequestHandler(dispatcher=dp, bot=bot).register(app, path=WEBHOOK_PATH)
```
Этот вызов связывает `aiogram` с Aiohttp, чтобы сервер мог передавать данные от Telegram API в `Dispatcher`.  
Когда Telegram отправляет данные на `WEBHOOK_PATH`, `SimpleRequestHandler` перенаправляет их в `Dispatcher`.

```python
app.on_startup.append(lambda _: asyncio.create_task(on_startup()))
app.on_shutdown.append(lambda _: asyncio.create_task(on_shutdown()))
```
Эти строки привязывают **асинхронные функции старта и остановки** к событиям Aiohttp:
- **`on_startup()`** вызывается при запуске, чтобы зарегистрировать webhook в Telegram API.
- **`on_shutdown()`** вызывается при завершении работы сервера, чтобы удалить webhook и корректно закрыть соединение.

`lambda _: asyncio.create_task(...)` используется, чтобы `on_startup()` и `on_shutdown()` выполнялись **асинхронно**, не блокируя Aiohttp.

```python
app.router.add_get("/", handle_root)
```
Добавляется **обработчик HTTP-запросов** для корневого пути `/`.  
Если кто-то откроет адрес `http://localhost:3000/` в браузере, сервер вернёт `"Сервер работает!"`.  
Этот маршрут не связан с Telegram, а просто служит для проверки, что сервер работает.

```python
runner = web.AppRunner(app)
await runner.setup()
site = web.TCPSite(runner, host=WEBAPP_HOST, port=WEBAPP_PORT)
```
- `web.AppRunner(app)` создаёт **менеджер сервера**, который управляет его запуском и остановкой.
- `await runner.setup()` подготавливает сервер к запуску.
- `site = web.TCPSite(runner, host=WEBAPP_HOST, port=WEBAPP_PORT)` создаёт TCP-сервер, который будет слушать входящие запросы на указанный порт.

```python
print(f"Сервер запущен на {WEBAPP_HOST}:{WEBAPP_PORT}")
await site.start()
```
- Выводится сообщение о том, что сервер запущен.
- `await site.start()` **запускает сервер и начинает принимать входящие запросы**.


### Сравнение polling и webhook

| Характеристика      | Polling                              | Webhook                              |
|---------------------|--------------------------------------|--------------------------------------|
| **Механизм**        | Клиент опрашивает сервер.            | Сервер уведомляет клиента.           |
| **Задержка**        | Зависит от интервала опроса.         | Почти мгновенно.                     |
| **Ресурсы**         | Высокая нагрузка на сервер и сеть.   | Меньшая нагрузка.                    |
| **Инфраструктура**  | Простая.                             | Требуется HTTPS и внешний сервер.    |


### Роль веб-сервера


В нашем коде веб-сервер — это Aiohttp. Он слушает входящие запросы и передаёт их Aiogram, который обрабатывает их как события от Telegram API.

#### **Как устроен веб-сервер в коде**
В коде веб-сервер создаётся так:
```python
app = web.Application()  # Создаём Aiohttp веб-приложение
runner = web.AppRunner(app)
await runner.setup()
site = web.TCPSite(runner, host=WEBAPP_HOST, port=WEBAPP_PORT)
await site.start()
```
Этот код запускает HTTP-сервер, который:
1. Слушает соединения на `WEBAPP_HOST:WEBAPP_PORT` (в нашем случае `0.0.0.0:3000`).
2. Принимает запросы от Telegram API на `/webhook`.
3. Передаёт их в `SimpleRequestHandler`, который перенаправляет их в Aiogram.

Обычно веб-приложения работают так:
```
[Клиент] → [Веб-сервер (Nginx)] → [WSGI/ASGI-сервер (Gunicorn/Uvicorn)] → [Python-приложение (FastAPI, Django)]
```
В нашем случае эта схема упрощена и выглядит так:
```
[Telegram API] → [Aiohttp (веб-сервер)] → [Aiogram (бот)]
```

### Классическая схема работы веб-приложения

#### Веб-сервер (Nginx, Apache)
- Это первый слой, принимающий входящие HTTP-запросы от клиентов.
- Может выполнять балансировку нагрузки, кеширование.
- Например, запросы от пользователей могут сначала прийти на Nginx, который передаст их дальше в приложение.

#### WSGI/ASGI-сервер (Gunicorn, Uvicorn)
- Этот слой отвечает за выполнение Python-приложения.
- WSGI (Web Server Gateway Interface) — стандарт взаимодействия между веб-сервером и Python-приложением (используется в Django, Flask).
- ASGI (Asynchronous Server Gateway Interface) — более современный стандарт, поддерживающий асинхронные фреймворки (FastAPI, Starlette).

#### Python-приложение
- Это основной код, который выполняет логику обработки запросов.
- Может быть написан на Flask, Django, FastAPI и т. д.

Пример работы запроса в классическом стеке:
1. Клиент делает запрос к `https://example.com/api/data`.
2. Nginx принимает запрос, выполняет кеширование или SSL-терминацию.
3. Gunicorn (WSGI) передаёт запрос в Python-приложение.
4. Приложение (Flask/Django/FastAPI) обрабатывает запрос и формирует ответ.
5. Gunicorn возвращает ответ Nginx, который отправляет его клиенту.

### Как это работает у нас
В нашем коде:

1. Мы не используем Nginx — запросы приходят напрямую в `Aiohttp`.
2. Мы не используем WSGI/ASGI-сервер — Aiohttp сам является асинхронным веб-сервером.
3. Aiogram — это Python-приложение, обрабатывающее запросы.

То есть у нас веб-сервер (`Aiohttp`) совмещён с Python-приложением (`Aiogram`):
```
[Telegram API] → [Aiohttp] → [Aiogram (бот)]
```
Запросы Telegram идут сразу в `Aiohttp`, который:
1. Принимает запрос (как Nginx).
2. Обрабатывает его асинхронно (как Gunicorn/Uvicorn).
3. Передаёт данные в `Aiogram`, который исполняет логику бота.

### Что у нас реализуется явно, что неявно, а что вообще отсутствует
| Компонент | В классической схеме | В нашем коде | Явно/Неявно/Отсутствует |
|-----------|---------------------|-------------|--------------------|
| Веб-сервер (Nginx, Apache) | Принимает запросы, балансирует нагрузку | Отсутствует, запросы идут сразу в Aiohttp | Отсутствует |
| WSGI/ASGI-сервер (Gunicorn, Uvicorn) | Запускает Python-приложение | Aiohttp сам управляет серверами | Неявно |
| Python-приложение (Flask, Django, FastAPI) | Обрабатывает HTTP-запросы | Aiogram обрабатывает запросы Telegram API | Явно |
| Webhook-обработчик | В Django/FastAPI через `@app.post("/webhook")` | `SimpleRequestHandler` передаёт запросы в Aiogram | Явно |

В обычной схеме Python-приложение не может само принимать HTTP-запросы, ему нужен WSGI/ASGI-сервер!

### FastAPI — это и фреймворк, и веб-сервер?



FastAPI — это асинхронный веб-фреймворк, но он не является веб-сервером. Он поддерживает ASGI (Asynchronous Server Gateway Interface), но для работы с сетью ему нужен ASGI-сервер (например, Uvicorn или Hypercorn).  

#### FastAPI ≠ веб-сервер
Когда мы запускаем приложение на FastAPI, мы не просто запускаем FastAPI, а используем Uvicorn для обработки HTTP-запросов.  
Стандартная схема:
```
[Клиент] → [Nginx] → [ASGI-сервер (Uvicorn)] → [FastAPI]
```
FastAPI сам по себе не умеет слушать HTTP-запросы, он только обрабатывает их после того, как их передаст сервер (Uvicorn).

Пример запуска FastAPI:
```bash
uvicorn main:app --host 0.0.0.0 --port 8000
```
Здесь:
- `uvicorn` — это **веб-сервер**.
- `main:app` — это наш FastAPI-приложение.
- `host=0.0.0.0` — сервер слушает все входящие соединения.
- `port=8000` — сервер запущен на порту 8000.

#### А другие?
- Django работает через WSGI и не может принимать HTTP-запросы напрямую.
- Стандартная схема:
  ```
  [Клиент] → [Nginx] → [Gunicorn (WSGI)] → [Django]
  ```

- Flask тоже не умеет слушать HTTP-запросы сам, ему нужен WSGI-сервер.
- Стандартная схема:
  ```
  [Клиент] → [Nginx] → [Gunicorn (WSGI)] → [Flask]
  ```
- FastAPI **по умолчанию не является веб-сервером**, но его можно запустить через встроенный Uvicorn:

```python
import uvicorn
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "Hello World"}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
```
В этом случае FastAPI сам запускает Uvicorn.

- Aiohttp — это и веб-сервер, и фреймворк, поэтому он не требует Uvicorn/Gunicorn. Можно просто написать:

```python
from aiohttp import web

async def handle(request):
    return web.Response(text="Hello, Aiohttp!")

app = web.Application()
app.router.add_get("/", handle)

web.run_app(app, host="0.0.0.0", port=8000)  # Aiohttp сам запускает сервер
```
Здесь `web.run_app()` уже сам запускает сервер, без Uvicorn/Gunicorn.


### Немного про Django

Django включает в себя WSGI, но только для локальной разработки.  

Когда запускаем Django **локально** с помощью:
```bash
python manage.py runserver
```
он использует встроенный WSGI-сервер, который запускает приложение и слушает HTTP-запросы.

Этот встроенный сервер не предназначен для продакшена, потому что:
- Он однопоточный и плохо справляется с нагрузкой.
- Не поддерживает параллельную обработку запросов.
- Не умеет балансировать нагрузку между процессами.

Django требует внешний WSGI-сервер в продакшене**
В продакшене Django не запускает сервер сам, а должен работать через Gunicorn или uWSGI.

Пример запуска Django через Gunicorn:
```bash
gunicorn myproject.wsgi:application --bind 0.0.0.0:8000
```
Здесь:
- `myproject.wsgi:application` — это WSGI-приложение Django.
- Gunicorn запускает его как многопоточный WSGI-сервер.

Файл `wsgi.py` в Django:
```python
import os
from django.core.wsgi import get_wsgi_application

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "myproject.settings")

application = get_wsgi_application()  # Это WSGI-приложение
```
Gunicorn или uWSGI загружают `application` и запускают Django внутри.

Django 3.1+ добавил поддержку ASGI.  
Если нужно асинхронное приложение (например, WebSocket), то запускаем Django через Uvicorn.

### Nginx

**Nginx** (произносится как "engine-x") — это незаменимый инструмент для современных веб-приложений. Он выступает как связующее звено между пользователем и сервером, обеспечивая эффективную обработку запросов и защиту серверной части.

#### Зачем нужен Nginx?

1. **Обслуживание статического контента**:
   - Nginx превосходно справляется с задачей доставки статических файлов (HTML, CSS, JS, изображения, видео). Это снимает нагрузку с приложения, позволяя серверу сосредоточиться на более сложных вычислениях.

2. **Обратный прокси**:
   - Nginx принимает запросы от пользователей и перенаправляет их на внутренние серверы (например, на Gunicorn, Flask, Django или Node.js). Это упрощает инфраструктуру и улучшает безопасность.

3. **Балансировка нагрузки**:
   - Для высоконагруженных систем Nginx распределяет запросы между несколькими серверами. Это позволяет справляться с большим количеством пользователей и увеличивает отказоустойчивость.

4. **Поддержка HTTPS**:
   - Nginx обеспечивает защиту данных, шифруя соединения через SSL/TLS. Это делает веб-сайты безопасными для пользователей и обязательным для SEO.

5. **Кеширование**:
   - Nginx может сохранять часто запрашиваемые ресурсы в памяти. Это ускоряет доступ к ним и снижает нагрузку на сервер.

6. **Ограничение доступа**:
   - С помощью Nginx можно ограничивать количество запросов с одного IP-адреса, предотвращая DDoS-атаки.

#### Почему без него нельзя?

1. **Плохая масштабируемость**:
   - Без Nginx сервер вашего приложения будет перегружен, так как ему придется обрабатывать все запросы: от статических файлов до сложных операций.

2. **Отсутствие безопасности**:
   - Прямое соединение пользователей с сервером приложения увеличивает риск утечек данных и атак. Nginx выполняет роль первого защитного слоя.

3. **Медленная работа**:
   - Nginx ускоряет доставку контента за счет кеширования, оптимизации соединений и сжатия данных. Без него загрузка сайта может быть медленной.

4. **Отказоустойчивость**:
   - Nginx помогает управлять трафиком и перенаправлять запросы в случае сбоя одного из серверов. Без него система становится менее устойчивой к нагрузкам.

### Вывод

**Nginx** делает веб-приложения быстрее, безопаснее и масштабируемее. Без него любая система становится уязвимой к перегрузкам, атакам и проблемам с производительностью. Даже небольшие проекты выигрывают от использования Nginx, поскольку это позволяет с самого начала закладывать основу для надежной и эффективной работы.

### Как работает стек FastAPI + Uvicorn + Gunicorn + Nginx  

#### 1. Архитектура  
В продакшене FastAPI обычно работает так:  
[Клиент] → [Nginx] → [Gunicorn/Uvicorn] → [FastAPI-приложение]  

1. Nginx принимает запросы, балансирует нагрузку, выполняет SSL-терминацию.  
2. Gunicorn + Uvicorn workers запускает FastAPI в многопроцессорном режиме.  
3. FastAPI обрабатывает логику приложения.  

Но локально можно запустить без Nginx:  
[Клиент] → [Uvicorn] → [FastAPI]  

#### 2. Установка зависимостей  

```bash
pip install fastapi uvicorn gunicorn
```  

#### 3. Пишем FastAPI-приложение (app.py)  

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
async def read_root():
    return {"message": "Hello, FastAPI!"}

@app.get("/items/{item_id}")
async def read_item(item_id: int, q: str = None):
    return {"item_id": item_id, "query": q}
```
Это простое API, которое:  
- Отвечает `{ "message": "Hello, FastAPI!" }` на `/`  
- Позволяет получать параметры запроса через `/items/{item_id}?q=somequery`  

#### 4. Запуск FastAPI напрямую через Uvicorn  
Запустим приложение локально без дополнительных серверов.  

```bash
uvicorn app:app --host 0.0.0.0 --port 8000 --reload
```  

Разбор флагов:  
- `app:app` — `имя_файла:объект_FastAPI`  
- `--host 0.0.0.0` — слушать все входящие соединения  
- `--port 8000` — сервер работает на `http://localhost:8000`  
- `--reload` — автоматическая перезагрузка при изменении кода  

Проверка работы:  
```bash
curl http://localhost:8000
```  
Должно вернуть:  
```json
{"message": "Hello, FastAPI!"}
```  

#### 5. Запуск FastAPI через Gunicorn с Uvicorn workers  
Uvicorn умеет работать сам, но если приложение должно обрабатывать много запросов, лучше запустить его через Gunicorn.  

```bash
gunicorn -w 4 -k uvicorn.workers.UvicornWorker -b 0.0.0.0:8000 app:app
```  

Разбор флагов:  
- `-w 4` — запустить 4 worker-процесса (по количеству ядер CPU)  
- `-k uvicorn.workers.UvicornWorker` — использовать Uvicorn внутри Gunicorn  
- `-b 0.0.0.0:8000` — привязаться к порту 8000  
- `app:app` — загружает FastAPI-приложение  

Проверка работы:  
```bash
curl http://localhost:8000
```

### Добавляем Nginx

In [ ]:
sudo -s

#### 1. Установка Nginx
Если Nginx не установлен, поставь его через Homebrew:
```bash
brew install nginx
```

Проверь, что он установился:
```bash
nginx -v
```
Если команда отработала и вывела версию Nginx — всё в порядке.

#### 2. Запуск и остановка Nginx
- Запустить:
  ```bash
  sudo brew services start nginx
  ```
- Остановить:
  ```bash
  sudo brew services stop nginx
  ```
- Перезапустить (если изменил конфиг):
  ```bash
  sudo brew services restart nginx
  ```
- Проверить статус:
  ```bash
  brew services list
  ```
- Проверить, запущен ли процесс:
  ```bash
  ps aux | grep nginx
  ```

После запуска можно проверить, работает ли Nginx, открыв в браузере:
```
http://localhost
```
По умолчанию он показывает страницу `Welcome to nginx!`.

#### 3. Где лежат конфиги Nginx на macOS
На macOS конфигурация Nginx находится здесь:
```
/opt/homebrew/etc/nginx/nginx.conf
```
Чтобы её изменить, открой файл:
```bash
sudo nano /opt/homebrew/etc/nginx/nginx.conf
```

По умолчанию в `nginx.conf` есть такой блок:
```nginx
server {
    listen 8080;
    server_name localhost;
    location / {
        root   html;
        index  index.html;
    }
}
```
Этот конфиг говорит, что Nginx слушает **порт 8080**.  

Но нам нужно, чтобы он работал с FastAPI через Gunicorn.

#### 4. Создаём конфиг для FastAPI
Создай директории для конфигурации (как на Linux):
```bash
sudo mkdir -p /opt/homebrew/etc/nginx/sites-available
sudo mkdir -p /opt/homebrew/etc/nginx/sites-enabled
```

Создай файл `/opt/homebrew/etc/nginx/sites-available/fastapi`:
```bash
sudo nano /opt/homebrew/etc/nginx/sites-available/fastapi
```

Добавь туда конфигурацию:
```nginx
server {
    listen 80;
    server_name localhost;

    location / {
        proxy_pass http://127.0.0.1:8000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
    }
}
```

Активируй этот конфиг:
```bash
sudo ln -s /opt/homebrew/etc/nginx/sites-available/fastapi /opt/homebrew/etc/nginx/sites-enabled/
```

Теперь открой `nginx.conf`:
```bash
sudo nano /opt/homebrew/etc/nginx/nginx.conf
```
Найди строку с `include servers/*;` и добавь перед ней:
```nginx
include /opt/homebrew/etc/nginx/sites-enabled/*;
```

#### 5. Проверяем и перезапускаем Nginx
- Проверяем конфигурацию на ошибки:
  ```bash
  sudo nginx -t
  ```
- Если ошибок нет, перезапускаем:
  ```bash
  sudo brew services restart nginx
  ```

Теперь Nginx проксирует запросы на **http://localhost/** к FastAPI.

### 6. Запуск FastAPI через Gunicorn
1. Запусти FastAPI с Gunicorn:
   ```bash
   gunicorn -w 4 -k uvicorn.workers.UvicornWorker -b 127.0.0.1:8000 app:app
   ```
2. Проверь работу:
   ```bash
   curl http://localhost/
   ```
   Должен вернуться ответ:
   ```json
   {"message": "Hello, FastAPI!"}
   ```

#### 7. Подключаем ngrok (если нужен внешний доступ)
1. Установи ngrok, если его нет:
   ```bash
   brew install ngrok
   ```
2. Запусти туннель на 80-й порт:
   ```bash
   ngrok http 80
   ```
3. После запуска появится публичный URL, например:
   ```
   https://12ab-34-567-89-123.ngrok-free.app -> http://localhost:80
   ```
4. Проверь, что FastAPI доступен через интернет:
   ```bash
   curl https://12ab-34-567-89-123.ngrok-free.app/
   ```


#### Отключить кастомные конфиги:
1. Удалить симлинки из `sites-enabled`:
   ```bash
   sudo rm /opt/homebrew/etc/nginx/sites-enabled/fastapi
   ```
2. Открыть `nginx.conf` и удалить строку, которую мы добавляли:
   ```bash
   sudo nano /opt/homebrew/etc/nginx/nginx.conf
   ```
   Удали эту строку:
   ```nginx
   include /opt/homebrew/etc/nginx/sites-enabled/*;
   ```
3. Перезапустить Nginx:
   ```bash
   sudo brew services restart nginx
   ```


#### Пара примеров:

Балансировка нагрузки с несколькими инстансами FastAPI

##### Шаги:

1. Запустите несколько инстансов FastAPI на разных портах:

   ```bash
   uvicorn app:app --host 127.0.0.1 --port 8001 &
   uvicorn app:app --host 127.0.0.1 --port 8002 &
   ```

2. Настройте Nginx для балансировки нагрузки:

   Откройте конфигурацию и добавьте:

   ```nginx
   upstream fastapi_backend {
       server 127.0.0.1:8001;
       server 127.0.0.1:8002;
   }

   server {
       listen 443 ssl;
       server_name localhost;

       ssl_certificate /path/to/localhost.pem;
       ssl_certificate_key /path/to/localhost-key.pem;

       location / {
           proxy_pass http://fastapi_backend;
           proxy_set_header Host $host;
           proxy_set_header X-Real-IP $remote_addr;
           proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
       }
   }
   ```

3. Перезапустите Nginx:

   ```bash
   brew services restart nginx
   ```

4. Проверьте:
   - Перейдите на [https://localhost](https://localhost).
   - Убедитесь, что запросы равномерно распределяются между портами 8001 и 8002.


Настройка кеширования

##### Шаги:

1. Добавьте кеширование для `/items`:

   ```nginx
   proxy_cache_path /var/cache/nginx levels=1:2 keys_zone=fastapi_cache:10m max_size=1g inactive=60m;

   server {
       listen 443 ssl;
       server_name localhost;

       ssl_certificate /path/to/localhost.pem;
       ssl_certificate_key /path/to/localhost-key.pem;

       location /items/ {
           proxy_cache fastapi_cache;
           proxy_pass http://fastapi_backend;
           add_header X-Cache-Status $upstream_cache_status;
       }

       location / {
           proxy_pass http://fastapi_backend;
       }
   }
   ```

2. Проверьте кеширование:
   - Отправьте запрос на `/items/1`.
   - Убедитесь, что повторный запрос быстрее и возвращает заголовок `X-Cache-Status: HIT`.
